# Interactive embedding renderer

Explore cached MNIST-family and arXiv layouts with Datashader. This notebook does not load high-dimensional data or refit embeddings. Run notebook 00 to create the MNIST cache; run 03a and 03 to prepare the arXiv vectors and coordinates.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pyproject.toml").exists() and (PROJECT_ROOT.parent / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "notebooks" / "src"))
sys.path.insert(0, str(PROJECT_ROOT))
from imste.visualization import render_datashader_embedding


## Load available coordinate caches

In [ ]:
DATASETS = {}
mnist_path = PROJECT_ROOT / "data" / "embeddings" / "mnist_gallery_embeddings.npz"
if mnist_path.is_file():
    with np.load(mnist_path, allow_pickle=False) as saved:
        for index, name in enumerate(saved["dataset_names"].astype(str)):
            DATASETS[name] = {
                "labels": saved[f"labels_{index}"],
                "UMAP": (saved[f"umap_{index}"], float(saved[f"umap_seconds_{index}"])),
                "IMSTE": (saved[f"imste_{index}"], float(saved[f"imste_seconds_{index}"])),
            }

arxiv_path = PROJECT_ROOT / "data" / "arxiv" / "cache" / "arxiv_embeddings.npz"
if arxiv_path.is_file():
    with np.load(arxiv_path, allow_pickle=False) as saved:
        labels = saved["labels"].astype(str)
        DATASETS[f"arXiv ({len(labels):,})"] = {
            "labels": labels,
            "UMAP": (saved["umap"], float(saved["umap_seconds"])),
            "IMSTE · UMAP start": (saved["imste_umap_start"], float(saved["imste_seconds"])),
            "IMSTE · multilevel start": (saved["imste_multilevel"], float(saved["multilevel_seconds"])),
        }

if not DATASETS:
    raise FileNotFoundError("No embedding caches found. Run notebook 00 for MNIST or notebooks 03a and 03 for arXiv.")
print("Available datasets:", ", ".join(DATASETS))


## Render a cached embedding

Choose a dataset and layout, then adjust the raster size, point spread, opacity, and background. Increasing point spread makes individual samples more visible; Datashader still aggregates into the selected canvas dimensions.

In [ ]:
dataset_widget = widgets.Dropdown(options=list(DATASETS), description="Dataset:", layout=widgets.Layout(width="50%"))
method_widget = widgets.Dropdown(description="Layout:")
width_widget = widgets.IntSlider(value=1100, min=400, max=2200, step=100, description="Width:", continuous_update=False)
height_widget = widgets.IntSlider(value=850, min=400, max=1800, step=100, description="Height:", continuous_update=False)
spread_widget = widgets.IntSlider(value=2, min=0, max=12, step=1, description="Spread px:", continuous_update=False)
alpha_widget = widgets.IntSlider(value=40, min=0, max=255, step=5, description="Min alpha:", continuous_update=False)
background_widget = widgets.Dropdown(options=["white", "black"], value="white", description="Background:")
render_button = widgets.Button(description="Render", button_style="primary", icon="image")
output = widgets.Output()


def update_methods(*_):
    method_widget.options = list(DATASETS[dataset_widget.value].keys())
    method_widget.value = method_widget.options[0]


def render_selected(_=None):
    selected = DATASETS[dataset_widget.value]
    embedding, seconds = selected[method_widget.value]
    image = render_datashader_embedding(
        embedding,
        selected["labels"],
        width=width_widget.value,
        height=height_widget.value,
        spread_px=spread_widget.value,
        min_alpha=alpha_widget.value,
        background=background_widget.value,
    )
    with output:
        clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(10, 8), constrained_layout=True)
        ax.imshow(image.to_pil())
        runtime = f"{seconds / 60:.1f} min" if seconds >= 60 else f"{seconds:.1f} s"
        ax.set_title(f"{dataset_widget.value} — {method_widget.value} (fit: {runtime})")
        ax.set_axis_off()
        display(fig)
        plt.close(fig)


dataset_widget.observe(update_methods, names="value")
render_button.on_click(render_selected)
update_methods()
display(widgets.VBox([
    dataset_widget, method_widget,
    widgets.HBox([width_widget, height_widget]),
    widgets.HBox([spread_widget, alpha_widget]),
    widgets.HBox([background_widget, render_button]),
    output,
]))
render_selected()
